In [71]:
import numpy as np
import pandas as pd
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains

In [72]:
def wait_for_page_to_load(driver,wait):
    title = driver.title
    try:
        wait.until(
            lambda d: d.execute_script("return document.readyState") == "complete"
        )
    except:
        print(f"The webpage \"{title}\" did not get fully loaded")
    else:
        print(f"Webpage: \"{title}\"; Fully Loaded !")


In [83]:
# options
chrome_options = Options()
chrome_options.add_argument("--disable-http2")
chrome_options.add_argument("--incognito")
chrome_options.add_argument("--disable-blink-features=AutomationControlled")
chrome_options.add_argument("--ignore-certificate-errors")
chrome_options.add_argument("--enable-features=NetworkServiceInProcess")
chrome_options.add_argument("--disable-features=NetworkService")
chrome_options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/93.0.4577.63 Safari/537.36"
)

driver = webdriver.Chrome(options=chrome_options)
driver.maximize_window()

# explicit wait
wait = WebDriverWait(driver,10)

# accessing the target webpage
url = "https://www.99acres.com/"
driver.get(url)
wait_for_page_to_load(driver,wait)

# identifying search_bar
try:
    search_bar = wait.until(
        EC.presence_of_element_located((By.XPATH,'//*[@id="keyword2"]'))
    )
except:
    print("Timeout while locating SearchBar.\n")
else:
    search_bar.send_keys("Chennai")
    time.sleep(2)

# selecting valid option from list
try:
    valid_option = wait.until(
        EC.element_to_be_clickable((By.XPATH,'//*[@id="0"]'))
    )
except:
    print("Timeout while locating valid search options.\n")
else:
    valid_option.click()
    time.sleep(2)

# click on search button
try:
    search_button = wait.until(
        EC.element_to_be_clickable((By.XPATH,'//*[@id="searchform_search_btn"]'))
    ) 
except:
    print("Timeout while clicking on the \"search\" button")
else:
    search_button.click()
    wait_for_page_to_load(driver,wait)

# adjusting the min value budget button 
try:
    filter_min = wait.until(
        EC.element_to_be_clickable((By.ID, "bdf__lfBudMin"))
    )
except:
    print("min_button_selector error")
else:
    filter_min.click()

try:
    filter_min_select_value = wait.until(
        EC.visibility_of_element_located((By.XPATH, "//*[@id='lf_budget_min_list']/li[3]"))
    )
except:
    print("min_button_select_value error")
else:
    filter_min_select_value.click()
    
# adjusting max value budget button
try:
    filter_max = wait.until(
        EC.element_to_be_clickable((By.ID, "bdf__lf_budMax"))
    )
except:
    print("min_button_selector error")
else:
    filter_max.click()

try:
    filter_max_select_value = wait.until(
        EC.visibility_of_element_located((By.XPATH, "//*[@id='lf_budget_max_list']/li[@data-val='17']"))
    )
except:
    print("max_button_select_value error")
else:
    filter_max_select_value.click()
    
# filter results to show genuine listings

# 1.Verified
verified = wait.until(EC.element_to_be_clickable(
    (By.CSS_SELECTOR, "label[for='verified_Y']")
))
verified.click()
time.sleep(2)


# 2. Ready to move
ready_to_move = wait.until(EC.element_to_be_clickable(
    (By.XPATH,'/html[1]/body[1]/div[1]/div[1]/div[1]/div[4]/div[3]/div[1]/div[3]/section[1]/div[1]/div[1]/div[1]/div[1]/div[1]/div[1]/div[5]/span[2]')
))
ready_to_move.click()
time.sleep(2)

# moving to the right side to unhide the remaining elements
while True:
    try:
        filter_right_button = wait.until(
            EC.presence_of_element_located((By.XPATH,'/html[1]/body[1]/div[1]/div[1]/div[1]/div[4]/div[3]/div[1]/div[3]/section[1]/div[1]/div[1]/div[1]/div[1]/div[2]/i[1]'))
        )
    except:
        print("Timeout because we have uncovered all filters. \n")
        break
    else:
        filter_right_button.click()
        time.sleep(2)

# 3. With  photos
with_photos = wait.until(
    EC.element_to_be_clickable((By.XPATH,'/html[1]/body[1]/div[1]/div[1]/div[1]/div[4]/div[3]/div[1]/div[3]/section[1]/div[1]/div[1]/div[1]/div[1]/div[2]/div[1]/div[6]/span[2]'))
)
with_photos.click()
time.sleep(1)

# 4. With videos
with_videos = wait.until(
    EC.element_to_be_clickable((By.XPATH,'/html[1]/body[1]/div[1]/div[1]/div[1]/div[4]/div[3]/div[1]/div[3]/section[1]/div[1]/div[1]/div[1]/div[1]/div[2]/div[1]/div[7]/span[2]'))
)
with_videos.click()
time.sleep(1)

# navigate pages and extract data
data = []
page_count = 0
while True:
    page_count += 1
    try:
        next_page_button = driver.find_element((By.XPATH,"//a[normalize-space()='Next Page >']"))
    except:
        print(f"Timeout because we have navigated all {page_count} pages.\n")
        break
    else:
        try:
             
            driver.execute_script("window.scrollBy(0,arguments[0].getBoundingClientRect().top-100);",next_page_button)
            time.sleep(2)
            # scraping the data
            rows = driver.find_elements(By.CLASS_NAME, "tupleNew__TupleContent")
            for row in rows:
                # property name
                try:
                    name = row.find_element(By.CLASS_NAME, "tupleNew__headingNrera").text
                except:
                    name = np.nan
                    
                # property location
                try:
                    location = row.find_element(By.CLASS_NAME, "tupleNew__propType").text
                except:
                    location = np.nan
                        
                # property price
                try:
                    price = row.find_element(By.CLASS_NAME, "tupleNew__priceValWrap").text
                except:
                    price = np.nan
                        
                # property area and bhk
                try:
                    elements = row.find_elements(By.CLASS_NAME, "tupleNew__area1Type")
                except:
                    area, bhk = [np.nan, np.nan]
                else:
                    area, bhk = [ele.text for ele in elements]
                property = {
                    "name": name,
                    "location": location,
                    "price": price,
                    "area": area,
                    "bhk": bhk
                }
                data.append(property)

            wait.until(
                EC.element_to_be_clickable((By.XPATH, "//a[normalize-space()='Next Page >']"))
            ).click()
            time.sleep(10)

        except:
            print("Timeout while clicking on \"Next Page\".\n")


# scraping data from the last page
rows = driver.find_elements(By.CLASS_NAME, "tupleNew__TupleContent")
for row in rows:
    # property name
    try:
        name = row.find_element(By.CLASS_NAME, "tupleNew__headingNrera").text
    except:
        name = np.nan

    # property location
    try:
        location = row.find_element(By.CLASS_NAME, "tupleNew__propType").text
    except:
        location = np.nan

    # property price
    try:
        price = row.find_element(By.CLASS_NAME, "tupleNew__priceValWrap").text
    except:
        price = np.nan

    # property area and bhk
    try:
        elements = row.find_elements(By.CLASS_NAME, "tupleNew__area1Type")
    except:
        area, bhk = [np.nan, np.nan]
    else:
        area, bhk = [ele.text for ele in elements]
                    
    property = {
        "name": name,
        "location": location,
        "price": price,
        "area": area,
        "bhk": bhk
    }
    data.append(property)

time.sleep(10)
driver.quit()

Webpage: "India Real Estate Property Site - Buy Sell Rent Properties Portal - 99acres.com"; Fully Loaded !
Webpage: "Property in Chennai - Real Estate in Chennai"; Fully Loaded !
Timeout because we have uncovered all filters. 

Timeout because we have navigated all 1 pages.

